
# 05 - Ablations: which part of the encoding is doing the work?

Runs on **CLUSTER with GCN only**, to keep the sweep inside a free-Colab session. Eight
variants, one seed each, resumable per variant.

The main grid (`M0`-`M4`) tells us *whether* the traversal encoding helps. It cannot tell us
*why*. Four questions are separated here:

1. **Radius $r$.** How far does the traversal need to reach? $r=1$ is the degenerate case:
   the rewired graph is the original graph and every edge is at distance 1, so the encoding
   carries no information and the model should fall back to the M0 baseline. That makes
   $r=1$ a built-in sanity check as much as an ablation.
2. **Encoder.** Is the sinusoidal map actually earning its place, or would a raw scalar, a
   one-hot, or a learned embedding table do as well?
3. **Injection depth.** Inject the encoding once at the first layer, or re-derive it at every
   layer?
4. **`rewire-only` - the attribution control.** The rewired graph with a *constant* edge
   feature. This variant has all the extra connectivity of M3 and none of the positional
   signal. If it matches M3, then the gain came from rewiring - from shortening paths and
   relieving over-squashing - and not from the traversal encoding at all. It is the single
   most informative row in this notebook, and the only one that can distinguish those two
   explanations.

In [ ]:
# Colab: torch ships preinstalled, torch-geometric does not.
!pip -q install torch-geometric

In [ ]:

# ============================== CONFIGURATION ==============================
ARCH, DATASET, TASK = "gcn", "CLUSTER", "node"
N_CLASSES, NODE_VOCAB = 6, 7

HIDDEN, LAYERS, HEADS, DROPOUT = 64, 4, 4, 0.0
LR, EPOCHS, BATCH_SIZE, PATIENCE = 1e-3, 40, 32, 12
PE_DIM = 16
SEEDS = [0]

# name           r  encoder        layerwise  what it isolates
VARIANTS = [
    dict(name="r1",             radius=1, encoder="sinusoidal", layerwise=False),
    dict(name="r2",             radius=2, encoder="sinusoidal", layerwise=False),
    dict(name="r3",             radius=3, encoder="sinusoidal", layerwise=False),
    dict(name="enc-raw",        radius=2, encoder="raw",        layerwise=False),
    dict(name="enc-onehot",     radius=2, encoder="onehot",     layerwise=False),
    dict(name="enc-learnable",  radius=2, encoder="learnable",  layerwise=False),
    dict(name="layerwise",      radius=2, encoder="sinusoidal", layerwise=True),
    dict(name="rewire-only",    radius=2, encoder="const",      layerwise=False),
]
print(f"{len(VARIANTS)} variants x {len(SEEDS)} seed(s) on {ARCH.upper()}/{DATASET}")

In [ ]:
import os, json, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# Results go to Drive so they survive a session; the datasets go to local disk because
# they are large and freely re-downloadable.
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RESULT_ROOT, DATA_ROOT, IN_COLAB = "/content/drive/MyDrive/trpe", "/content/trpe_data", True
except Exception:
    RESULT_ROOT, DATA_ROOT = os.path.abspath("./trpe"), os.path.abspath("./trpe_data")

RESULTS_DIR = os.path.join(RESULT_ROOT, "results", "ablations")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
print("results ->", RESULTS_DIR)

In [ ]:

def compute_trpe(data, r):
    """Identical to the leaf notebooks: (A^k)>0 first at k gives d(u,v)."""
    n = int(data.num_nodes); ei = data.edge_index.numpy()
    A = np.zeros((n, n)); A[ei[0], ei[1]] = 1.0
    A = np.maximum(A, A.T); np.fill_diagonal(A, 0.0)
    dist = np.full((n, n), -1, dtype=np.int64); np.fill_diagonal(dist, 0)
    Mk = np.eye(n)
    for k in range(1, r + 1):
        Mk = Mk @ A
        fresh = (Mk > 0.5) & (dist < 0)
        if not fresh.any(): break
        dist[fresh] = k
    src, dst = np.nonzero(dist >= 1)
    data.edge_index = torch.from_numpy(np.stack([src, dst])).long()
    data.pe_dist = torch.from_numpy(dist[src, dst]).long()
    data.num_nodes = n
    return data


class SinusoidalDistance(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = max(1, dim // 2)
        self.register_buffer("freqs", torch.exp(-math.log(10000.0) * torch.arange(half).float() / half))
    def forward(self, d):
        ang = d.float().unsqueeze(-1) * self.freqs
        return torch.cat([ang.sin(), ang.cos()], dim=-1)


class EdgePE(nn.Module):
    """The four encoder variants, plus the constant 'rewire-only' control."""
    def __init__(self, kind, pe_dim, radius):
        super().__init__()
        self.kind = kind
        if kind == "sinusoidal":
            self.enc = SinusoidalDistance(pe_dim); in_dim = 2 * max(1, pe_dim // 2)
        elif kind == "raw":
            self.enc = None; in_dim = 1
        elif kind == "onehot":
            self.enc = None; in_dim = radius + 1
        elif kind == "learnable":
            self.enc = nn.Embedding(radius + 1, pe_dim); in_dim = pe_dim
        elif kind == "const":
            self.enc = None; in_dim = 1          # carries NO distance information
        self.radius = radius
        self.mlp = nn.Sequential(nn.Linear(in_dim, pe_dim), nn.ReLU(), nn.Linear(pe_dim, pe_dim))

    def forward(self, d):
        if self.kind == "sinusoidal":   e = self.enc(d)
        elif self.kind == "raw":        e = d.float().unsqueeze(-1) / max(1, self.radius)
        elif self.kind == "onehot":     e = F.one_hot(d.clamp(0, self.radius), self.radius + 1).float()
        elif self.kind == "learnable":  e = self.enc(d.clamp(0, self.radius))
        else:                           e = torch.ones_like(d, dtype=torch.float).unsqueeze(-1)
        return self.mlp(e)


class Net(nn.Module):
    def __init__(self, in_dim, out_dim, cfg):
        super().__init__()
        n_enc = LAYERS if cfg["layerwise"] else 1
        self.edge_pes = nn.ModuleList(
            [EdgePE(cfg["encoder"], PE_DIM, cfg["radius"]) for _ in range(n_enc)])
        self.gates = nn.ModuleList([nn.Linear(PE_DIM, 1) for _ in range(n_enc)])
        self.layerwise = cfg["layerwise"]
        self.embed = nn.Linear(in_dim, HIDDEN)
        self.convs = nn.ModuleList([GCNConv(HIDDEN, HIDDEN) for _ in range(LAYERS)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(HIDDEN) for _ in range(LAYERS)])
        self.head = nn.Linear(HIDDEN, out_dim)

    def forward(self, data):
        x = self.embed(data.x)
        shared = None if self.layerwise else torch.sigmoid(
            self.gates[0](self.edge_pes[0](data.pe_dist))).squeeze(-1)
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            w = shared if shared is not None else torch.sigmoid(
                self.gates[i](self.edge_pes[i](data.pe_dist))).squeeze(-1)
            x = x + F.relu(norm(conv(x, data.edge_index, edge_weight=w)))
        return self.head(x)

In [ ]:

# ---------------------------------- data ----------------------------------
def make_transform(radius):
    def t(data):
        if data.x.dim() == 1:
            data.x = F.one_hot(data.x.long().clamp(0, NODE_VOCAB - 1), NODE_VOCAB).float()
        data.x = data.x.float()
        return compute_trpe(data, radius)
    return t


def loaders(radius, seed):
    torch.manual_seed(seed)
    root = os.path.join(DATA_ROOT, DATASET)
    t = make_transform(radius)
    tr = GNNBenchmarkDataset(root, DATASET, split="train", transform=t)
    va = GNNBenchmarkDataset(root, DATASET, split="val",   transform=t)
    te = GNNBenchmarkDataset(root, DATASET, split="test",  transform=t)
    return (DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
            DataLoader(va, batch_size=BATCH_SIZE, num_workers=2),
            DataLoader(te, batch_size=BATCH_SIZE, num_workers=2), len(tr))


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for b in loader:
        b = b.to(DEVICE)
        y = b.y.view(-1)
        correct += int((model(b).argmax(-1) == y).sum()); total += y.numel()
    return correct / max(1, total)


def run_variant(cfg, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te, n_train = loaders(cfg["radius"], seed)
    torch.manual_seed(seed)
    model = Net(NODE_VOCAB, N_CLASSES, cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, bad, t0 = {"val_acc": -1, "test_acc": 0.0}, 0, time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for b in tr:
            b = b.to(DEVICE); opt.zero_grad()
            F.cross_entropy(model(b), b.y.view(-1)).backward(); opt.step()
        v = evaluate(model, va)
        if v > best["val_acc"]:
            best = {"epoch": epoch, "val_acc": v, "test_acc": evaluate(model, te)}; bad = 0
        else:
            bad += 1
        if epoch % 5 == 0:
            print(f"    ep {epoch:3d} | val {v:.4f} | best {best['val_acc']:.4f} -> test {best['test_acc']:.4f}")
        if bad >= PATIENCE:
            break
    return {**best, "seconds": round(time.time() - t0, 1),
            "params": sum(p.numel() for p in model.parameters())}

In [ ]:

# ---------------------------------- run ----------------------------------
results = []
for cfg in VARIANTS:
    for seed in SEEDS:
        path = os.path.join(RESULTS_DIR, f"{cfg['name']}_seed{seed}.json")
        if os.path.exists(path):
            print(f"{cfg['name']} seed{seed}: already done, skipping")
            results.append(json.load(open(path))); continue
        print(f"\n=== {cfg['name']} (r={cfg['radius']}, {cfg['encoder']}, "
              f"layerwise={cfg['layerwise']}) seed {seed} ===")
        res = run_variant(cfg, seed)
        res.update(variant=cfg["name"], seed=seed, arch=ARCH, dataset=DATASET, **cfg)
        json.dump(res, open(path, "w"), indent=2)
        results.append(res)
        print(f"  -> test {res['test_acc']:.4f}  ({res['seconds']}s)")

print(f"\n{'variant':<16}{'r':>3}{'encoder':>13}{'layerwise':>11}{'test acc':>11}")
print("-" * 54)
for r in results:
    print(f"{r['variant']:<16}{r['radius']:>3}{r['encoder']:>13}{str(r['layerwise']):>11}"
          f"{100*r['test_acc']:>10.2f}%")


## How to read this table

* **`r1` should land on the M0 baseline.** At $r=1$ the rewired graph *is* the original
  graph and every edge carries distance 1, so there is nothing for the encoding to say. If
  `r1` beats M0 by a real margin, something is leaking and the rest of the results cannot be
  trusted.
* **`r2` vs `r3`** trades signal against cost. The edge count, and therefore the training
  time, grows sharply with $r$ (see `06_efficiency.ipynb`); a flat curve means the extra
  reach is not worth paying for.
* **`rewire-only` is the row that decides the paper's claim.** If it matches `r2`, the
  benefit is connectivity rather than position, and the honest conclusion is that our
  contribution is a rewiring scheme, not a positional encoding. If it sits clearly below
  `r2`, the distance signal is doing real work. Either outcome is reportable; only the
  comparison makes the main result interpretable.
* **Encoder rows** answer whether the sinusoidal map is load-bearing. If `enc-onehot` and
  `enc-learnable` match it, say so - with $r\le 3$ there are only three distinct distances,
  so a smooth encoding has little room to help, and claiming otherwise would overstate.